# SCBF Training on Google Colab

**Setup:** Runtime > Change runtime type > GPU (T4 or better)

**Dataset:** Upload SCBF folder with data inside

## Step 1: Install Dependencies

In [ ]:
!pip install torch numpy jsonlines scikit-learn matplotlib -q
print("✓ Dependencies installed")

## Step 2: Verify Dataset

In [ ]:
import glob
import json

# Check if SCBF folder exists
import os
if not os.path.exists('SCBF'):
    print("⚠ SCBF folder not found!")
    print("")
    print("Upload your SCBF folder to Colab:")
    print("1. Click folder icon (left sidebar)")
    print("2. Upload the entire SCBF folder from your computer")
    print("3. Re-run this cell after upload completes")
else:
    benign_files = glob.glob("SCBF/data/zenodo_13746167/benign/traces/*.jsonl")
    malware_files = glob.glob("SCBF/data/zenodo_13746167/malware/traces/*.jsonl")
    
    print(f"Found {len(benign_files)} benign packages")
    print(f"Found {len(malware_files)} malware packages")
    print(f"Total: {len(benign_files) + len(malware_files)} packages")
    
    # Test load one file
    if len(benign_files) > 0:
        with open(benign_files[0], 'r') as f:
            events = [json.loads(line) for line in f]
        print(f"\nSample file has {len(events)} events")
        print(f"First event: {events[0]}")
    
    assert len(benign_files) > 0, "No benign files found!"
    assert len(malware_files) > 0, "No malware files found!"
    print("\n✓ Data verified!")

## Step 3: Check GPU

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print("\n✓ GPU training enabled - will be FAST!")
else:
    print("\n⚠ Using CPU - training will be slower")
    print("Enable GPU: Runtime > Change runtime type > GPU")

## Step 4: Train Model

**GPU:** ~30-60 minutes  
**CPU:** ~2-3 hours

In [ ]:
# Change to SCBF directory
%cd SCBF

# Create checkpoint directory
!mkdir -p models/checkpoints

# Run training
!python -m scbf.training.train_with_split

## Step 5: Evaluate Model

In [ ]:
import torch
import json
import numpy as np
import sys

# Make sure we're in SCBF directory
%cd /content/SCBF
sys.path.insert(0, '/content/SCBF')

from scbf.models.tgn_encoder import TGNEncoder
from scbf.models.itbg_constructor import ITBGConstructor

# Load split info
with open("models/checkpoints/split_info.json", 'r') as f:
    split_info = json.load(f)

test_data = split_info['test']

print(f"Test set: {len(test_data)} samples")
print(f"  Clean: {sum(1 for x in test_data if x['label'] == 0)}")
print(f"  Malicious: {sum(1 for x in test_data if x['label'] == 1)}")

# Load model
print("\nLoading model...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TGNEncoder(num_nodes=50000).to(device)
model.load_state_dict(torch.load("models/tgn_v2_best.pt", map_location=device))
model.eval()

def load_events(path):
    with open(path, 'r') as f:
        return [json.loads(line) for line in f]

# Compute embeddings
print("\nComputing embeddings...")
embeddings_list = []
labels_list = []

with torch.no_grad():
    for i, item in enumerate(test_data):
        if (i + 1) % 50 == 0:
            print(f"  Processed {i + 1}/{len(test_data)}...")
        
        path = item['path']
        label = item['label']
        
        model.memory_bank.reset_memory()
        constructor = ITBGConstructor(model)
        
        try:
            events = load_events(path)
            dna = constructor.replay_session(events)
            
            if dna is not None:
                embeddings_list.append(dna.cpu().numpy())
                labels_list.append(label)
        except Exception as e:
            continue

embeddings = np.array(embeddings_list)
labels = np.array(labels_list)

# Compute distances
clean_mask = labels == 0
clean_centroid = embeddings[clean_mask].mean(axis=0)
distances = np.sqrt(((embeddings - clean_centroid) ** 2).sum(axis=1))

clean_dists = distances[labels == 0]
mal_dists = distances[labels == 1]

print("\n" + "=" * 80)
print("DISTANCE STATISTICS")
print("=" * 80)
print(f"Clean distances: mean={clean_dists.mean():.4f}, std={clean_dists.std():.4f}")
print(f"Malicious distances: mean={mal_dists.mean():.4f}, std={mal_dists.std():.4f}")
print(f"Separation: {mal_dists.mean() - clean_dists.mean():.4f}")

# Auto-threshold
threshold = clean_dists.mean() + 2 * clean_dists.std()
print(f"\nAuto-computed threshold: {threshold:.4f}")

# Predictions
predictions = (distances > threshold).astype(int)

# Metrics
tp = ((predictions == 1) & (labels == 1)).sum()
tn = ((predictions == 0) & (labels == 0)).sum()
fp = ((predictions == 1) & (labels == 0)).sum()
fn = ((predictions == 0) & (labels == 1)).sum()

accuracy = (tp + tn) / len(labels)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print("\n" + "=" * 80)
print("TEST SET RESULTS")
print("=" * 80)
print(f"\nAccuracy:  {accuracy:.2%} ({accuracy:.4f})")
print(f"Precision: {precision:.2%} ({precision:.4f})")
print(f"Recall:    {recall:.2%} ({recall:.4f})")
print(f"F1 Score:  {f1:.2%} ({f1:.4f})")

print(f"\nConfusion Matrix:")
print(f"                Predicted")
print(f"              Clean  Malicious")
print(f"Actual Clean    {tn:3d}  {fp:3d}")
print(f"Actual Mal      {fn:3d}  {tp:3d}")

# AUC
try:
    from sklearn.metrics import roc_auc_score
    auc = roc_auc_score(labels, distances)
    print(f"\nROC-AUC: {auc:.4f}")
except:
    pass

print("\n" + "=" * 80)

# Summary
print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
if mal_dists.mean() > 1.5:
    print("✓ Good separation! Malicious distance > 1.5")
else:
    print("⚠ Weak separation. Malicious distance < 1.5")
    
if accuracy > 0.85:
    print("✓ Excellent accuracy > 85%")
elif accuracy > 0.80:
    print("✓ Good accuracy > 80%")
else:
    print("⚠ Accuracy needs improvement")
    
if recall > 0.70:
    print("✓ Good recall > 70%")
else:
    print("⚠ Low recall - missing malware detections")

print("=" * 80)

## Step 6: Download Trained Model

In [ ]:
from google.colab import files

# Download best model
files.download('models/tgn_v2_best.pt')

print("\n✓ Model downloaded!")
print("\nPlace it in: /Users/shield/Downloads/scbf/models/tgn_v2_best.pt")